In [9]:
"""
Credit Scoring Dashboard
Aplicação Streamlit para análise e predição de crédito
Desenvolvido para: Quod DataTech
"""

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score
)

# Função KS
def ks_statistic(y_true, y_pred_proba):
    """Calcula KS-Statistic"""
    data = pd.DataFrame({"y": y_true, "p": y_pred_proba}).sort_values("p")
    cum_pos = np.cumsum(data["y"] == 1) / (data["y"] == 1).sum()
    cum_neg = np.cumsum(data["y"] == 0) / (data["y"] == 0).sum()
    return np.max(np.abs(cum_pos - cum_neg))

# Configuração da página
st.set_page_config(
    page_title="Credit Scoring Dashboard",
    page_icon="💳",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Sidebar
st.sidebar.title("🎯 Navegação")
page = st.sidebar.radio(
    "Selecione a página:",
    ["🏠 Visão Geral", "📊 Performance do Modelo", "🔮 Fazer Predição", 
     "📈 Análise de Features", "💰 Impacto de Negócio"]
)

# Carregar dados e modelo
@st.cache_data
def load_data():
    test_df = pd.read_csv('../data/final/no_scale_test.csv')
    results_df = pd.read_csv('../reports/model_results.csv')
    return test_df, results_df

@st.cache_resource
def load_model():
    import glob
    model_files = glob.glob('../models/*FINAL.pkl')
    if model_files:
        model = joblib.load(model_files[0])
        model_name = model_files[0].split('/')[-1]
    else:
        model = joblib.load('../models/xgboost.pkl')
        model_name = 'xgboost.pkl'
    return model, model_name

test_df, results_df = load_data()
model, model_name = load_model()

# Variáveis principais
target_col = 'inadipl_90dias_ult2anos'
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
ks = ks_statistic(y_test, y_pred_proba)

baseline_auc = results_df[results_df['model_name'].str.contains('Baseline')]['test_auc'].values[0]

# ============================================================================
# PÁGINA 1: VISÃO GERAL
# ============================================================================
if page == "🏠 Visão Geral":
    st.title("💳 Credit Scoring Dashboard")
    st.markdown("### Análise Preditiva de Inadimplência")
    st.markdown("---")

    # Métricas principais
    col1, col2, col3, col4, col5 = st.columns(5)
    with col1:
        st.metric("AUC-ROC", f"{auc:.4f}", f"+{((auc - baseline_auc)/baseline_auc)*100:.1f}% vs Baseline")
    with col2:
        st.metric("KS-Statistic", f"{ks:.4f}", "Benchmark ≥ 0.30")
    with col3:
        st.metric("Accuracy", f"{accuracy*100:.2f}%")
    with col4:
        st.metric("F1-Score", f"{f1:.4f}")
    with col5:
        st.metric("Total Predições", f"{len(y_test):,}")

    st.markdown("---")
    st.subheader("🎯 Sobre o Projeto")
    st.markdown("""
    Este projeto desenvolveu um **modelo de Machine Learning** para predição 
    de inadimplência, auxiliando na tomada de decisão de concessão de crédito.

    **Destaques:**
    - ✅ 60+ features criadas via feature engineering
    - ✅ 4 algoritmos testados e comparados
    - ✅ Modelo final otimizado (LightGBM Tuned)
    - ✅ KS = 0.56 (excepcional poder discriminativo)
    - ✅ Análise de impacto financeiro
    """)

    st.markdown("---")
    st.subheader("📈 Evolução dos Modelos")
    results_sorted = results_df.sort_values('test_auc')
    fig = go.Figure(go.Bar(
        x=results_sorted['test_auc'],
        y=results_sorted['model_name'],
        orientation='h',
        text=results_sorted['test_auc'].round(4),
        textposition='auto',
        marker=dict(color=results_sorted['test_auc'], colorscale='Viridis')
    ))
    fig.update_layout(title="Comparação de Modelos - AUC-ROC", xaxis_title="AUC", height=400)
    st.plotly_chart(fig, width='stretch')

    st.markdown("---")
    st.subheader("📊 Distribuição do Dataset")
    cm = confusion_matrix(y_test, y_pred)
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=['Predito: Bom', 'Predito: Mau'],
        y=['Real: Bom', 'Real: Mau'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text}'
    ))
    fig.update_layout(title="Confusion Matrix", height=350)
    st.plotly_chart(fig, use_container_width=True)
# ============================================================================
# PÁGINA 2: PERFORMANCE DO MODELO
# ============================================================================
elif page == "📊 Performance do Modelo":
    st.title("📊 Performance do Modelo")
    st.markdown("### Análise Detalhada de Métricas")
    st.markdown("---")

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f'Modelo (AUC={auc:.4f}, KS={ks:.4f})',
                             line=dict(color='blue', width=3)))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                             name='Random', line=dict(color='red', dash='dash')))
    fig.update_layout(title="ROC Curve", xaxis_title="False Positive Rate",
                      yaxis_title="True Positive Rate", height=400)
    st.plotly_chart(fig, width='stretch')

    # Precision-Recall Curve
    from sklearn.metrics import precision_recall_curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
    fig = go.Figure(go.Scatter(x=recall_vals, y=precision_vals, mode='lines',
                               line=dict(color='green', width=3)))
    fig.update_layout(title="Precision-Recall Curve",
                      xaxis_title="Recall", yaxis_title="Precision", height=400)
    st.plotly_chart(fig, width='stretch')

# ============================================================================
# PÁGINA 3: FAZER PREDIÇÃO
# ============================================================================
elif page == "🔮 Fazer Predição":
    st.title("🔮 Simulador de Crédito")
    st.markdown("### Faça uma predição de inadimplência")
    st.markdown("---")

    with st.form("prediction_form"):
        idade = st.number_input("Idade", min_value=18, max_value=100, value=35)
        renda = st.number_input("Renda Mensal (R$)", min_value=0.0, value=5000.0, step=100.0)
        divida_renda = st.slider("Debt Ratio", 0.0, 2.0, 0.3, 0.01)
        linhas_credito = st.number_input("Linhas de Crédito Abertas", 0, 50, 5)
        atrasos = st.number_input("Total de Atrasos", 0, 20, 0)
        submitted = st.form_submit_button("🎯 Fazer Predição")

    if submitted:
        # Exemplo simplificado
        input_df = pd.DataFrame({
            'age':[idade],
            'MonthlyIncome':[renda],
            'DebtRatio':[divida_renda],
            'NumberOfOpenCreditLinesAndLoans':[linhas_credito],
            'TotalPastDue':[atrasos]
        })

        st.warning("⚠️ Este é um exemplo simplificado. O modelo real requer todas as features de feature engineering.")

        prob = model.predict_proba(input_df)[:,1][0] if hasattr(model, "predict_proba") else np.random.rand()
        st.metric("Probabilidade de Inadimplência", f"{prob*100:.2f}%")

        if prob < 0.3:
            st.success("✅ Aprovar: Cliente de baixo risco")
        elif prob < 0.7:
            st.warning("⚠️ Análise Manual: Risco moderado")
        else:
            st.error("❌ Rejeitar: Alto risco de inadimplência")
# ============================================================================
# PÁGINA 4: ANÁLISE DE FEATURES
# ============================================================================

elif page == "📈 Análise de Features":
    st.title("📈 Análise de Features")
    st.markdown("### Importância e Distribuição das Variáveis")
    st.markdown("---")
    
    target_col = 'SeriousDlqin2yrs'
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]
    
    # Feature Importance
    st.subheader("🎯 Feature Importance")
    
    feature_importance = pd.DataFrame({
        'feature': X_test.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Slider para número de features
    n_features = st.slider(
        "Número de features a exibir:",
        min_value=5,
        max_value=min(30, len(feature_importance)),
        value=15
    )
    
    top_features = feature_importance.head(n_features)
    
    fig = go.Figure(go.Bar(
        x=top_features['importance'],
        y=top_features['feature'],
        orientation='h',
        marker=dict(
            color=top_features['importance'],
            colorscale='Viridis',
            showscale=True
        ),
        text=top_features['importance'].round(4),
        textposition='auto'
    ))
    
    fig.update_layout(
        title=f"Top {n_features} Features Mais Importantes",
        xaxis_title="Importância",
        yaxis_title="Feature",
        height=max(400, n_features * 25),
        yaxis={'categoryorder': 'total ascending'}
    )
    
    st.plotly_chart(fig, use_container_width=True)
    
    st.markdown("---")
    
    # Análise de distribuição
    st.subheader("📊 Distribuição das Features")
    
    # Seletor de feature
    selected_feature = st.selectbox(
        "Selecione uma feature para análise:",
        options=X_test.columns.tolist()
    )
    
    col1, col2 = st.columns(2)
    
    with col1:
        # Histograma por classe
        fig = go.Figure()
        
        fig.add_trace(go.Histogram(
            x=test_df[test_df[target_col] == 0][selected_feature],
            name='Bom Pagador',
            opacity=0.7,
            marker=dict(color='green'),
            nbinsx=30
        ))
        
        fig.add_trace(go.Histogram(
            x=test_df[test_df[target_col] == 1][selected_feature],
            name='Inadimplente',
            opacity=0.7,
            marker=dict(color='red'),
            nbinsx=30
        ))
        
        fig.update_layout(
            title=f"Distribuição de {selected_feature}",
            xaxis_title=selected_feature,
            yaxis_title="Frequência",
            barmode='overlay',
            height=400
        )
        
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        # Box plot por classe
        fig = go.Figure()
        
        fig.add_trace(go.Box(
            y=test_df[test_df[target_col] == 0][selected_feature],
            name='Bom Pagador',
            marker=dict(color='green')
        ))
        
        fig.add_trace(go.Box(
            y=test_df[test_df[target_col] == 1][selected_feature],
            name='Inadimplente',
            marker=dict(color='red')
        ))
        
        fig.update_layout(
            title=f"Box Plot de {selected_feature}",
            yaxis_title=selected_feature,
            height=400
        )
        
        st.plotly_chart(fig, use_container_width=True)
    
    # Estatísticas descritivas
    st.subheader("📋 Estatísticas Descritivas")
    
    stats_bom = test_df[test_df[target_col] == 0][selected_feature].describe()
    stats_mau = test_df[test_df[target_col] == 1][selected_feature].describe()
    
    stats_df = pd.DataFrame({
        'Bom Pagador': stats_bom,
        'Inadimplente': stats_mau,
        'Diferença': stats_mau - stats_bom
    })
    
    st.dataframe(stats_df.style.background_gradient(cmap='RdYlGn_r', axis=1), 
                use_container_width=True)
    
    # Correlação com target
    correlation = test_df[[selected_feature, target_col]].corr().iloc[0, 1]
    
    if abs(correlation) > 0.3:
        strength = "forte"
        color = "red"
    elif abs(correlation) > 0.1:
        strength = "moderada"
        color = "orange"
    else:
        strength = "fraca"
        color = "green"
    
    st.info(f"""
    **Correlação com Inadimplência:** {correlation:+.4f}  
    A feature tem correlação **{strength}** com inadimplência.
    """)
    
    st.markdown("---")
    
    # Matriz de correlação (top features)
    st.subheader("🔗 Matriz de Correlação (Top 10 Features)")
    
    top_10_features = feature_importance.head(10)['feature'].tolist()
    corr_matrix = test_df[top_10_features + [target_col]].corr()
    
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=corr_matrix.round(2),
        texttemplate='%{text}',
        textfont={"size": 10}
    ))
    
    fig.update_layout(
        title="Matriz de Correlação",
        height=600,
        width=600
    )
    
    st.plotly_chart(fig, use_container_width=True)

# ============================================================================
# PÁGINA 5: IMPACTO DE NEGÓCIO
# ============================================================================

elif page == "💰 Impacto de Negócio":
    st.title("💰 Impacto de Negócio")
    st.markdown("### Análise Financeira e ROI")
    st.markdown("---")
    
    target_col = 'SeriousDlqin2yrs'
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Configurações financeiras
    st.subheader("⚙️ Configurações Financeiras")
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        valor_medio_credito = st.number_input(
            "Valor Médio de Crédito (R$)",
            min_value=1000.0,
            max_value=100000.0,
            value=5000.0,
            step=500.0
        )
    
    with col2:
        taxa_perda = st.slider(
            "Taxa de Perda em Inadimplência (%)",
            min_value=0,
            max_value=100,
            value=70
        ) / 100
    
    with col3:
        custo_oportunidade = st.number_input(
            "Custo de Oportunidade - Cliente Rejeitado (R$)",
            min_value=0.0,
            max_value=5000.0,
            value=500.0,
            step=100.0
        )
    
    # Threshold ajustável
    st.markdown("---")
    st.subheader("🎚️ Otimização de Threshold")
    
    business_threshold = st.slider(
        "Selecione o threshold de decisão:",
        min_value=0.0,
        max_value=1.0,
        value=0.5,
        step=0.05,
        help="Ajuste o threshold para maximizar o retorno financeiro"
    )
    
    # Calcular métricas com threshold selecionado
    y_pred_business = (y_pred_proba >= business_threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_business)
    tn, fp, fn, tp = cm.ravel()
    
    # Cálculos financeiros
    perda_fn = fn * valor_medio_credito * taxa_perda
    perda_fp = fp * custo_oportunidade
    economia_tp = tp * valor_medio_credito * taxa_perda
    ganho_tn = tn * custo_oportunidade * 0.1  # Pequeno ganho por aprovar corretamente
    
    impacto_total = economia_tp + ganho_tn - perda_fn - perda_fp
    
    # Baseline (modelo aleatório)
    total_inadimplentes = (y_test == 1).sum()
    random_tp = total_inadimplentes * 0.5
    random_economia = random_tp * valor_medio_credito * taxa_perda
    random_fn = total_inadimplentes - random_tp
    random_perda_fn = random_fn * valor_medio_credito * taxa_perda
    random_impacto = random_economia - random_perda_fn
    
    ganho_vs_random = impacto_total - random_impacto
    
    # Exibir métricas
    st.markdown("---")
    st.subheader("💵 Impacto Financeiro Estimado")
    
    col1, col2, col3, col4 = st.columns(4)
    
    with col1:
        st.metric(
            label="💰 Economia (TP)",
            value=f"R$ {economia_tp:,.2f}",
            delta=f"{tp:,} inadimplentes detectados",
            delta_color="normal"
        )
    
    with col2:
        st.metric(
            label="💸 Perda (FN)",
            value=f"R$ {perda_fn:,.2f}",
            delta=f"{fn:,} inadimplentes não detectados",
            delta_color="inverse"
        )
    
    with col3:
        st.metric(
            label="🚫 Oportunidade Perdida (FP)",
            value=f"R$ {perda_fp:,.2f}",
            delta=f"{fp:,} bons clientes rejeitados",
            delta_color="inverse"
        )
    
    with col4:
        st.metric(
            label="📊 Impacto Líquido",
            value=f"R$ {impacto_total:,.2f}",
            delta=f"+R$ {ganho_vs_random:,.2f} vs Random",
            delta_color="normal"
        )
    
    # Gráfico de waterfall
    st.markdown("---")
    
    fig = go.Figure(go.Waterfall(
        name="Impacto",
        orientation="v",
        measure=["relative", "relative", "relative", "relative", "total"],
        x=["Economia (TP)", "Perda (FN)", "Oportunidade<br>Perdida (FP)", "Ganho (TN)", "Total"],
        y=[economia_tp, -perda_fn, -perda_fp, ganho_tn, impacto_total],
        text=[f"R$ {economia_tp:,.0f}", f"R$ {perda_fn:,.0f}", 
              f"R$ {perda_fp:,.0f}", f"R$ {ganho_tn:,.0f}", f"R$ {impacto_total:,.0f}"],
        textposition="outside",
        connector={"line": {"color": "rgb(63, 63, 63)"}},
        decreasing={"marker": {"color": "#e74c3c"}},
        increasing={"marker": {"color": "#2ecc71"}},
        totals={"marker": {"color": "#3498db"}}
    ))
    
    fig.update_layout(
        title=f"Análise de Impacto Financeiro (Threshold: {business_threshold})",
        showlegend=False,
        height=500
    )
    
    st.plotly_chart(fig, use_container_width=True)
    
    st.markdown("---")
    
    # Análise por threshold
    st.subheader("📈 Otimização de Threshold para Máximo Retorno")
    
    thresholds_range = np.arange(0.1, 0.9, 0.05)
    impacts = []
    
    for thresh in thresholds_range:
        y_pred_t = (y_pred_proba >= thresh).astype(int)
        cm_t = confusion_matrix(y_test, y_pred_t)
        tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
        
        perda_fn_t = fn_t * valor_medio_credito * taxa_perda
        perda_fp_t = fp_t * custo_oportunidade
        economia_tp_t = tp_t * valor_medio_credito * taxa_perda
        ganho_tn_t = tn_t * custo_oportunidade * 0.1
        
        impact_t = economia_tp_t + ganho_tn_t - perda_fn_t - perda_fp_t
        
        impacts.append({
            'threshold': thresh,
            'impacto': impact_t,
            'fn': fn_t,
            'fp': fp_t,
            'tp': tp_t
        })
    
    impacts_df = pd.DataFrame(impacts)
    optimal_idx = impacts_df['impacto'].idxmax()
    optimal_threshold = impacts_df.loc[optimal_idx, 'threshold']
    optimal_impact = impacts_df.loc[optimal_idx, 'impacto']
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=impacts_df['threshold'],
        y=impacts_df['impacto'],
        mode='lines+markers',
        name='Impacto Financeiro',
        line=dict(color='blue', width=3),
        marker=dict(size=8)
    ))
    
    fig.add_vline(
        x=optimal_threshold,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Ótimo: {optimal_threshold:.2f}",
        annotation_position="top"
    )
    
    fig.add_vline(
        x=business_threshold,
        line_dash="dash",
        line_color="green",
        annotation_text=f"Atual: {business_threshold:.2f}",
        annotation_position="bottom"
    )
    
    fig.update_layout(
        title="Impacto Financeiro por Threshold",
        xaxis_title="Threshold",
        yaxis_title="Impacto Financeiro (R$)",
        height=500,
        hovermode='x'
    )
    
    st.plotly_chart(fig, use_container_width=True)
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.success(f"""
        **🎯 Threshold Ótimo Identificado:**  
        Threshold: **{optimal_threshold:.2f}**  
        Impacto: **R$ {optimal_impact:,.2f}**  
        
        Este threshold maximiza o retorno financeiro com base nos parâmetros informados.
        """)
    
    with col2:
        diff = optimal_impact - impacto_total
        if diff > 0:
            st.warning(f"""
            **💡 Oportunidade de Melhoria:**  
            Ajustando para o threshold ótimo, você poderia ganhar mais:  
            **+R$ {diff:,.2f}**
            """)
        else:
            st.info(f"""
            **✅ Threshold Atual é Ótimo!**  
            Você já está usando (ou próximo de) o threshold que maximiza o retorno.
            """)
    
    st.markdown("---")
    
    # ROI Anual
    st.subheader("📊 Projeção Anual")
    
    col1, col2 = st.columns(2)
    
    with col1:
        volume_mensal = st.number_input(
            "Volume Mensal de Análises",
            min_value=100,
            max_value=100000,
            value=1000,
            step=100
        )
    
    with col2:
        custo_implementacao = st.number_input(
            "Custo de Implementação (R$)",
            min_value=0.0,
            max_value=1000000.0,
            value=50000.0,
            step=5000.0
        )
    
    # Calcular projeção
    impacto_por_analise = impacto_total / len(y_test)
    impacto_mensal = impacto_por_analise * volume_mensal
    impacto_anual = impacto_mensal * 12
    roi_anual = ((impacto_anual - custo_implementacao) / custo_implementacao) * 100
    payback_meses = custo_implementacao / impacto_mensal if impacto_mensal > 0 else float('inf')
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric(
            "💰 Impacto Anual",
            f"R$ {impacto_anual:,.2f}"
        )
    
    with col2:
        st.metric(
            "📈 ROI Anual",
            f"{roi_anual:.1f}%"
        )
    
    with col3:
        st.metric(
            "⏱️ Payback",
            f"{payback_meses:.1f} meses" if payback_meses != float('inf') else "N/A"
        )
    
    # Gráfico de projeção
    meses = list(range(1, 13))
    impacto_acumulado = [impacto_mensal * m for m in meses]
    custo_acumulado = [custo_implementacao] * 12
    lucro_acumulado = [imp - custo_implementacao for imp in impacto_acumulado]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=meses,
        y=impacto_acumulado,
        mode='lines+markers',
        name='Impacto Acumulado',
        line=dict(color='green', width=3)
    ))
    
    fig.add_trace(go.Scatter(
        x=meses,
        y=custo_acumulado,
        mode='lines',
        name='Custo de Implementação',
        line=dict(color='red', width=2, dash='dash')
    ))
    
    fig.add_trace(go.Scatter(
        x=meses,
        y=lucro_acumulado,
        mode='lines+markers',
        name='Lucro Líquido',
        line=dict(color='blue', width=3),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title="Projeção Financeira Anual",
        xaxis_title="Mês",
        yaxis_title="Valor (R$)",
        height=500,
        hovermode='x unified'
    )
    
    st.plotly_chart(fig, use_container_width=True)
    
    # Resumo executivo
    st.markdown("---")
    st.subheader("📋 Resumo Executivo")
    
    st.markdown(f"""
    ### Análise de Viabilidade do Projeto
    
    **Investimento Inicial:** R$ {custo_implementacao:,.2f}
    
    **Retorno Esperado:**
    - Impacto Mensal: R$ {impacto_mensal:,.2f}
    - Impacto Anual: R$ {impacto_anual:,.2f}
    - ROI em 12 meses: {roi_anual:.1f}%
    - Período de Payback: {payback_meses:.1f} meses
    
    **Benefícios Principais:**
    - ✅ Redução de perdas por inadimplência
    - ✅ Otimização da taxa de aprovação
    - ✅ Decisões baseadas em dados
    - ✅ Processo automatizado e escalável
    
    **Recomendação:** {'✅ PROJETO VIÁVEL' if roi_anual > 100 else '⚠️ AVALIAR CUIDADOSAMENTE' if roi_anual > 50 else '❌ NÃO RECOMENDADO'}
    """)

# ============================================================================
# FOOTER
# ============================================================================

st.markdown("---")
st.markdown("""
<div style='text-align: center; color: #7f8c8d; padding: 20px;'>
    <p><strong>Credit Scoring Dashboard</strong></p>
    <p>Desenvolvido para Quod DataTech | Projeto de Machine Learning</p>
    <p>📧 [Seu Email] | 💼 [Seu LinkedIn] | 🐙 [Seu GitHub]</p>
</div>
""", unsafe_allow_html=True)

2026-03-06 11:07:44.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.165 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-06 11:07:44.167 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

SyntaxError: invalid syntax (2907776460.py, line 5)